In [1]:
from pathlib import Path
import pandas as pd
import duckdb

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

FEATURES = PROJECT_ROOT / "data" / "features"
RESULTS = PROJECT_ROOT / "results"

BEHAVIOR_FILE = (
    FEATURES /
    "behavior_features_r62_v1.parquet"
)

OUTPUT_FILE = (
    RESULTS /
    "feature_dictionary_r62.csv"
)

RESULTS.mkdir(parents=True, exist_ok=True)

behavior_path = str(
    BEHAVIOR_FILE
).replace("\\", "/")

assert BEHAVIOR_FILE.exists()

print("Input :", BEHAVIOR_FILE)
print("Output:", OUTPUT_FILE)

print(
    "\n✅ SUCCESS — Feature Dictionary setup ready."
)

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Input : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\behavior_features_r62_v1.parquet
Output: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\results\feature_dictionary_r62.csv

✅ SUCCESS — Feature Dictionary setup ready.


In [3]:
# =========================================================
# SEMANTIC FEATURE DEFINITIONS
# =========================================================

records = []


def add(
    feature_name,
    category,
    feature_role,
    description,
    calculation,
    time_scope,
    source_columns,
    notes=""
):
    records.append({
        "feature_name": feature_name,
        "category": category,
        "feature_role": feature_role,
        "description": description,
        "calculation": calculation,
        "time_scope": time_scope,
        "source_columns": source_columns,
        "notes": notes
    })


# =========================================================
# 1. IDENTIFIERS / CALENDAR
# =========================================================

add(
    "user",
    "Identifier",
    "Key",
    "Unique identifier of the user.",
    "Original user identifier.",
    "User level",
    "user"
)

add(
    "day",
    "Calendar",
    "Key",
    "Calendar day used to aggregate the user's activity.",
    "CAST(timestamp AS DATE)",
    "Daily",
    "date / timestamp"
)

add(
    "day_of_week",
    "Calendar",
    "Context",
    "Numeric day of the week, where Monday = 1 and Sunday = 7.",
    "strftime(day, '%u')",
    "Daily",
    "day"
)

add(
    "is_weekend",
    "Calendar",
    "Context",
    "Indicates whether the day is Saturday or Sunday.",
    "1 if day_of_week in (6,7), else 0",
    "Daily",
    "day_of_week",
    "The behavior dataset contains weekday workdays only, so this value is normally 0."
)


# =========================================================
# 2. EMAIL DAILY FEATURES
# =========================================================

add(
    "email_event_count",
    "Email",
    "Observed Daily",
    "Total number of email events generated by the user during the day.",
    "COUNT(email events)",
    "Daily",
    "email.csv"
)

add(
    "email_send_count",
    "Email",
    "Observed Daily",
    "Number of email Send events during the day.",
    "COUNT(activity = 'Send')",
    "Daily",
    "email.activity"
)

add(
    "email_view_count",
    "Email",
    "Observed Daily",
    "Number of email View events during the day.",
    "COUNT(activity = 'View')",
    "Daily",
    "email.activity"
)

add(
    "external_email_count",
    "Email",
    "Observed Daily",
    "Number of sent emails containing at least one recipient outside the @dtaa.com domain.",
    "COUNT(external Send events)",
    "Daily",
    "to, cc, bcc, activity",
    "Recipients are parsed individually and empty recipient values are ignored."
)

add(
    "internal_email_count",
    "Email",
    "Observed Daily",
    "Number of sent emails classified as internal.",
    "email_send_count - external_email_count",
    "Daily",
    "to, cc, bcc, activity"
)

add(
    "total_sent_email_bytes",
    "Email",
    "Observed Daily",
    "Total size in bytes of emails sent by the user during the day.",
    "SUM(size) for Send events",
    "Daily",
    "email.size"
)

add(
    "external_email_bytes",
    "Email",
    "Observed Daily",
    "Total size in bytes of external emails sent during the day.",
    "SUM(size) for external Send events",
    "Daily",
    "email.size, recipients"
)

add(
    "emails_with_attachments",
    "Email",
    "Observed Daily",
    "Number of sent emails containing one or more attachments.",
    "COUNT(sent emails with attachments)",
    "Daily",
    "email.attachments"
)

add(
    "attachment_count",
    "Email",
    "Observed Daily",
    "Total number of attachments observed in emails during the day.",
    "SUM(parsed attachment count)",
    "Daily",
    "email.attachments"
)

add(
    "unique_external_recipients",
    "Email",
    "Observed Daily",
    "Number of unique external email recipients contacted during the day.",
    "COUNT(DISTINCT external recipient)",
    "Daily",
    "to, cc, bcc"
)

add(
    "unique_external_domains",
    "Email",
    "Observed Daily",
    "Number of unique external email domains contacted during the day.",
    "COUNT(DISTINCT external email domain)",
    "Daily",
    "external recipients"
)


# =========================================================
# 3. FILE / USB DAILY FEATURES
# =========================================================

add(
    "file_event_count",
    "USB/File",
    "Observed Daily",
    "Total number of file-system events generated by the user during the day.",
    "COUNT(file events)",
    "Daily",
    "file.csv"
)

add(
    "file_copy_count",
    "USB/File",
    "Observed Daily",
    "Number of File Copy events during the day.",
    "COUNT(activity = 'File Copy')",
    "Daily",
    "file.activity"
)

add(
    "file_open_count",
    "USB/File",
    "Observed Daily",
    "Number of File Open events during the day.",
    "COUNT(activity = 'File Open')",
    "Daily",
    "file.activity"
)

add(
    "file_write_count",
    "USB/File",
    "Observed Daily",
    "Number of File Write events during the day.",
    "COUNT(activity = 'File Write')",
    "Daily",
    "file.activity"
)

add(
    "file_delete_count",
    "USB/File",
    "Observed Daily",
    "Number of File Delete events during the day.",
    "COUNT(activity = 'File Delete')",
    "Daily",
    "file.activity"
)

add(
    "usb_copy_count",
    "USB/File",
    "Observed Daily",
    "Number of files copied to removable media during the day.",
    "COUNT(File Copy AND to_removable_media = true)",
    "Daily",
    "activity, to_removable_media"
)

add(
    "from_removable_count",
    "USB/File",
    "Observed Daily",
    "Number of file events involving files originating from removable media.",
    "COUNT(from_removable_media = true)",
    "Daily",
    "from_removable_media",
    "This includes all relevant file-event types, not only File Copy."
)

add(
    "unique_usb_files",
    "USB/File",
    "Observed Daily",
    "Number of unique filenames copied to removable media during the day.",
    "COUNT(DISTINCT filename) for USB copies",
    "Daily",
    "filename, to_removable_media"
)

add(
    "unique_usb_extensions",
    "USB/File",
    "Observed Daily",
    "Number of unique file extensions copied to removable media during the day.",
    "COUNT(DISTINCT file extension)",
    "Daily",
    "filename, to_removable_media"
)


# =========================================================
# 4. HTTP DAILY FEATURES
# =========================================================

add(
    "http_event_count",
    "HTTP",
    "Observed Daily",
    "Total number of HTTP events generated by the user during the day.",
    "COUNT(http events)",
    "Daily",
    "http.csv"
)

add(
    "www_visit_count",
    "HTTP",
    "Observed Daily",
    "Number of WWW Visit events during the day.",
    "COUNT(activity = 'WWW Visit')",
    "Daily",
    "http.activity"
)

add(
    "www_download_count",
    "HTTP",
    "Observed Daily",
    "Number of WWW Download events during the day.",
    "COUNT(activity = 'WWW Download')",
    "Daily",
    "http.activity"
)

add(
    "www_upload_count",
    "HTTP",
    "Observed Daily",
    "Number of WWW Upload events during the day.",
    "COUNT(activity = 'WWW Upload')",
    "Daily",
    "http.activity"
)

add(
    "unique_domains",
    "HTTP",
    "Observed Daily",
    "Number of unique web domains accessed during the day.",
    "COUNT(DISTINCT extracted domain)",
    "Daily",
    "http.url"
)

add(
    "unique_upload_domains",
    "HTTP",
    "Observed Daily",
    "Number of unique web domains receiving WWW Upload activity.",
    "COUNT(DISTINCT domain WHERE activity = 'WWW Upload')",
    "Daily",
    "url, activity"
)


# =========================================================
# 5. LOGON DAILY FEATURES
# =========================================================

add(
    "logon_count",
    "Logon",
    "Observed Daily",
    "Number of Logon events generated by the user during the day.",
    "COUNT(activity = 'Logon')",
    "Daily",
    "logon.activity"
)

add(
    "logoff_count",
    "Logon",
    "Observed Daily",
    "Number of Logoff events generated by the user during the day.",
    "COUNT(activity = 'Logoff')",
    "Daily",
    "logon.activity"
)

add(
    "first_logon",
    "Logon",
    "Observed Daily",
    "Timestamp of the user's first Logon event during the day.",
    "MIN(timestamp WHERE activity = 'Logon')",
    "Daily",
    "logon.date"
)

add(
    "last_logoff",
    "Logon",
    "Observed Daily",
    "Timestamp of the user's final Logoff event during the day.",
    "MAX(timestamp WHERE activity = 'Logoff')",
    "Daily",
    "logon.date"
)


# =========================================================
# 6. USER-SPECIFIC BASELINE FEATURES
# =========================================================

baseline_scope = (
    "Baseline period before 2011-01-01; "
    "weekdays with logon_count > 0"
)

add(
    "baseline_workdays",
    "Baseline",
    "User Baseline",
    "Number of workdays used to construct the user's behavioral baseline.",
    "COUNT(weekday user-days with logon_count > 0 before cutoff)",
    baseline_scope,
    "day, is_weekend, logon_count"
)

add(
    "email_active_days",
    "Baseline",
    "User Baseline",
    "Number of baseline workdays containing at least one external email.",
    "COUNT(external_email_count > 0)",
    baseline_scope,
    "external_email_count"
)

add(
    "email_active_rate",
    "Baseline",
    "User Baseline",
    "Proportion of baseline workdays on which the user sent external email.",
    "email_active_days / baseline_workdays",
    baseline_scope,
    "email_active_days, baseline_workdays"
)

for percentile in [50, 90, 95]:
    add(
        f"email_count_p{percentile}",
        "Baseline",
        "User Baseline",
        f"{percentile}th percentile of external email count on active external-email days.",
        f"QUANTILE_CONT(external_email_count, {percentile/100})",
        baseline_scope + "; active external-email days only",
        "external_email_count",
        "Zero-activity days are excluded from the distribution."
    )

add(
    "email_count_mad",
    "Baseline",
    "User Baseline",
    "Median absolute deviation of external email counts on active days.",
    "MEDIAN(ABS(external_email_count - email_count_p50))",
    baseline_scope + "; active external-email days only",
    "external_email_count, email_count_p50"
)

for percentile in [50, 90, 95]:
    add(
        f"email_bytes_p{percentile}",
        "Baseline",
        "User Baseline",
        f"{percentile}th percentile of external email bytes on active external-email days.",
        f"QUANTILE_CONT(external_email_bytes, {percentile/100})",
        baseline_scope + "; active external-email days only",
        "external_email_bytes"
    )

add(
    "email_bytes_mad",
    "Baseline",
    "User Baseline",
    "Median absolute deviation of external email bytes on active external-email days.",
    "MEDIAN(ABS(external_email_bytes - email_bytes_p50))",
    baseline_scope + "; active external-email days only",
    "external_email_bytes, email_bytes_p50"
)

add(
    "usb_active_days",
    "Baseline",
    "User Baseline",
    "Number of baseline workdays containing at least one USB copy.",
    "COUNT(usb_copy_count > 0)",
    baseline_scope,
    "usb_copy_count"
)

add(
    "usb_active_rate",
    "Baseline",
    "User Baseline",
    "Proportion of baseline workdays containing USB copy activity.",
    "usb_active_days / baseline_workdays",
    baseline_scope,
    "usb_active_days, baseline_workdays"
)

for percentile in [50, 90, 95]:
    add(
        f"usb_copy_p{percentile}",
        "Baseline",
        "User Baseline",
        f"{percentile}th percentile of USB copy count on active USB days.",
        f"QUANTILE_CONT(usb_copy_count, {percentile/100})",
        baseline_scope + "; active USB days only",
        "usb_copy_count"
    )

add(
    "usb_copy_mad",
    "Baseline",
    "User Baseline",
    "Median absolute deviation of USB copy counts on active USB days.",
    "MEDIAN(ABS(usb_copy_count - usb_copy_p50))",
    baseline_scope + "; active USB days only",
    "usb_copy_count, usb_copy_p50"
)

add(
    "http_upload_active_days",
    "Baseline",
    "User Baseline",
    "Number of baseline workdays containing at least one WWW Upload event.",
    "COUNT(www_upload_count > 0)",
    baseline_scope,
    "www_upload_count"
)

add(
    "http_upload_active_rate",
    "Baseline",
    "User Baseline",
    "Proportion of baseline workdays containing WWW Upload activity.",
    "http_upload_active_days / baseline_workdays",
    baseline_scope,
    "http_upload_active_days, baseline_workdays"
)

for percentile in [50, 90, 95]:
    add(
        f"http_upload_p{percentile}",
        "Baseline",
        "User Baseline",
        f"{percentile}th percentile of WWW Upload count on active upload days.",
        f"QUANTILE_CONT(www_upload_count, {percentile/100})",
        baseline_scope + "; active HTTP-upload days only",
        "www_upload_count"
    )

add(
    "http_upload_mad",
    "Baseline",
    "User Baseline",
    "Median absolute deviation of WWW Upload counts on active upload days.",
    "MEDIAN(ABS(www_upload_count - http_upload_p50))",
    baseline_scope + "; active HTTP-upload days only",
    "www_upload_count, http_upload_p50"
)


# =========================================================
# 7. CURRENT-DAY BEHAVIOR FEATURES
# =========================================================

future_scope = (
    "Future period from 2011-01-01 onward; "
    "weekday workdays with logon_count > 0"
)

add(
    "email_active_today",
    "Behavior",
    "Daily Derived",
    "Indicates whether external email activity occurred on the current day.",
    "1 if external_email_count > 0 else 0",
    future_scope,
    "external_email_count"
)

add(
    "usb_active_today",
    "Behavior",
    "Daily Derived",
    "Indicates whether USB copy activity occurred on the current day.",
    "1 if usb_copy_count > 0 else 0",
    future_scope,
    "usb_copy_count"
)

add(
    "http_upload_active_today",
    "Behavior",
    "Daily Derived",
    "Indicates whether WWW Upload activity occurred on the current day.",
    "1 if www_upload_count > 0 else 0",
    future_scope,
    "www_upload_count"
)

add(
    "active_channel_count",
    "Cross-Channel",
    "Daily Derived",
    "Number of active outbound channels among external email, USB copy, and HTTP upload.",
    "email_active_today + usb_active_today + http_upload_active_today",
    future_scope,
    "email_active_today, usb_active_today, http_upload_active_today",
    "Range: 0 to 3."
)


# =========================================================
# 8. BASELINE-RELATIVE P95 FLAGS
# =========================================================

add(
    "email_count_above_p95",
    "Baseline-Relative",
    "Anomaly Indicator",
    "Indicates whether today's external email count exceeds the user's historical P95.",
    "1 if external_email_count > email_count_p95 else 0",
    future_scope,
    "external_email_count, email_count_p95"
)

add(
    "email_bytes_above_p95",
    "Baseline-Relative",
    "Anomaly Indicator",
    "Indicates whether today's external email volume exceeds the user's historical P95.",
    "1 if external_email_bytes > email_bytes_p95 else 0",
    future_scope,
    "external_email_bytes, email_bytes_p95"
)

add(
    "usb_copy_above_p95",
    "Baseline-Relative",
    "Anomaly Indicator",
    "Indicates whether today's USB copy count exceeds the user's historical P95.",
    "1 if usb_copy_count > usb_copy_p95 else 0",
    future_scope,
    "usb_copy_count, usb_copy_p95"
)

add(
    "http_upload_above_p95",
    "Baseline-Relative",
    "Anomaly Indicator",
    "Indicates whether today's WWW Upload count exceeds the user's historical P95.",
    "1 if www_upload_count > http_upload_p95 else 0",
    future_scope,
    "www_upload_count, http_upload_p95"
)


# =========================================================
# 9. BASELINE-RELATIVE P95 RATIOS
# =========================================================

add(
    "email_count_to_p95_ratio",
    "Baseline-Relative",
    "Relative Magnitude",
    "Ratio of today's external email count to the user's historical P95.",
    "external_email_count / email_count_p95",
    future_scope,
    "external_email_count, email_count_p95",
    "NULL when the user's P95 is missing or zero."
)

add(
    "email_bytes_to_p95_ratio",
    "Baseline-Relative",
    "Relative Magnitude",
    "Ratio of today's external email volume to the user's historical P95.",
    "external_email_bytes / email_bytes_p95",
    future_scope,
    "external_email_bytes, email_bytes_p95",
    "NULL when the user's P95 is missing or zero."
)

add(
    "usb_copy_to_p95_ratio",
    "Baseline-Relative",
    "Relative Magnitude",
    "Ratio of today's USB copy count to the user's historical P95.",
    "usb_copy_count / usb_copy_p95",
    future_scope,
    "usb_copy_count, usb_copy_p95",
    "NULL when the user's P95 is missing or zero."
)

add(
    "http_upload_to_p95_ratio",
    "Baseline-Relative",
    "Relative Magnitude",
    "Ratio of today's WWW Upload count to the user's historical P95.",
    "www_upload_count / http_upload_p95",
    future_scope,
    "www_upload_count, http_upload_p95",
    "NULL when the user's P95 is missing or zero."
)


# =========================================================
# 10. TEMPORAL ACTIVE-DAY FEATURES
# =========================================================

for window in [5, 10, 20]:

    add(
        f"email_active_days_{window}wd",
        "Temporal",
        "Rolling Behavior",
        f"Number of workdays with external email activity within the most recent {window} user workdays, including the current day.",
        f"SUM(email_active_today) over last {window} user workday rows",
        f"Rolling {window} workdays including current day",
        "email_active_today",
        "wd means workdays, not weeks or calendar days."
    )

    add(
        f"usb_active_days_{window}wd",
        "Temporal",
        "Rolling Behavior",
        f"Number of workdays with USB copy activity within the most recent {window} user workdays, including the current day.",
        f"SUM(usb_active_today) over last {window} user workday rows",
        f"Rolling {window} workdays including current day",
        "usb_active_today",
        "wd means workdays."
    )

    add(
        f"http_active_days_{window}wd",
        "Temporal",
        "Rolling Behavior",
        f"Number of workdays with HTTP upload activity within the most recent {window} user workdays, including the current day.",
        f"SUM(http_upload_active_today) over last {window} user workday rows",
        f"Rolling {window} workdays including current day",
        "http_upload_active_today",
        "wd means workdays."
    )


# =========================================================
# 11. TEMPORAL ACTIVITY COUNTS
# =========================================================

for window in [5, 10]:

    add(
        f"external_email_count_{window}wd",
        "Temporal",
        "Rolling Volume",
        f"Total external email count across the most recent {window} user workdays, including the current day.",
        f"SUM(external_email_count) over last {window} workday rows",
        f"Rolling {window} workdays including current day",
        "external_email_count"
    )

    add(
        f"usb_copy_count_{window}wd",
        "Temporal",
        "Rolling Volume",
        f"Total USB copy count across the most recent {window} user workdays, including the current day.",
        f"SUM(usb_copy_count) over last {window} workday rows",
        f"Rolling {window} workdays including current day",
        "usb_copy_count"
    )

    add(
        f"http_upload_count_{window}wd",
        "Temporal",
        "Rolling Volume",
        f"Total WWW Upload count across the most recent {window} user workdays, including the current day.",
        f"SUM(www_upload_count) over last {window} workday rows",
        f"Rolling {window} workdays including current day",
        "www_upload_count"
    )


# =========================================================
# 12. CROSS-CHANNEL TEMPORAL FEATURE
# =========================================================

add(
    "multi_channel_days_10wd",
    "Cross-Channel",
    "Rolling Behavior",
    "Number of days within the most recent 10 user workdays where at least two outbound channels were active.",
    "SUM(active_channel_count >= 2) over last 10 workday rows",
    "Rolling 10 workdays including current day",
    "active_channel_count",
    "Captures repeated cross-channel behavior rather than transfer size."
)


feature_metadata = pd.DataFrame(records)

print(
    "Semantic definitions created:",
    len(feature_metadata)
)

print(
    "\n✅ SUCCESS — Semantic feature definitions created."
)

Semantic definitions created: 85

✅ SUCCESS — Semantic feature definitions created.


In [5]:
# =========================================================
# FEATURE DICTIONARY COVERAGE VALIDATION
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{behavior_path}'
)
""").df()

con.close()


actual_features = set(
    schema["column_name"]
)

documented_features = set(
    feature_metadata["feature_name"]
)


missing_documentation = (
    actual_features
    - documented_features
)

extra_documentation = (
    documented_features
    - actual_features
)

duplicate_definitions = (
    feature_metadata[
        feature_metadata[
            "feature_name"
        ].duplicated(keep=False)
    ]
)


print("=" * 100)
print("FEATURE DICTIONARY COVERAGE VALIDATION")
print("=" * 100)

print(
    "Actual behavior columns     :",
    len(actual_features)
)

print(
    "Documented feature columns :",
    len(documented_features)
)

print("\nMissing documentation:")

print(
    sorted(missing_documentation)
    if missing_documentation
    else "None"
)

print("\nExtra definitions:")

print(
    sorted(extra_documentation)
    if extra_documentation
    else "None"
)

print(
    "\nDuplicate definitions:",
    len(duplicate_definitions)
)


assert len(actual_features) == 85
assert len(documented_features) == 85

assert len(missing_documentation) == 0
assert len(extra_documentation) == 0
assert len(duplicate_definitions) == 0


print(
    "\n✅ SUCCESS — All 85 behavior columns are documented exactly once."
)

FEATURE DICTIONARY COVERAGE VALIDATION
Actual behavior columns     : 85
Documented feature columns : 85

Missing documentation:
None

Extra definitions:
None

Duplicate definitions: 0

✅ SUCCESS — All 85 behavior columns are documented exactly once.


In [7]:
# =========================================================
# BUILD FINAL FEATURE DICTIONARY
# =========================================================

schema_clean = (
    schema[
        [
            "column_name",
            "column_type",
            "null"
        ]
    ]
    .copy()
)


# Preserve the exact column order
# from behavior_features_r62_v1.parquet
schema_clean.insert(
    0,
    "column_order",
    range(
        1,
        len(schema_clean) + 1
    )
)


schema_clean = schema_clean.rename(
    columns={
        "column_name": "feature_name",
        "column_type": "data_type",
        "null": "nullable"
    }
)


feature_dictionary = (
    schema_clean
    .merge(
        feature_metadata,
        on="feature_name",
        how="left",
        validate="one_to_one"
    )
)


# Add dataset name
feature_dictionary.insert(
    1,
    "dataset",
    "behavior_features_r62_v1"
)


print("=" * 120)
print("FINAL FEATURE DICTIONARY")
print("=" * 120)

print(
    feature_dictionary[
        [
            "column_order",
            "feature_name",
            "category",
            "feature_role",
            "data_type",
            "description"
        ]
    ].to_string(index=False)
)

print(
    "\nTotal documented columns:",
    len(feature_dictionary)
)

assert len(feature_dictionary) == 85

print(
    "\n✅ SUCCESS — Final Feature Dictionary assembled."
)

FINAL FEATURE DICTIONARY
 column_order               feature_name          category       feature_role data_type                                                                                                         description
            1                       user        Identifier                Key   VARCHAR                                                                                      Unique identifier of the user.
            2                        day          Calendar                Key      DATE                                                                 Calendar day used to aggregate the user's activity.
            3                day_of_week          Calendar            Context   INTEGER                                                           Numeric day of the week, where Monday = 1 and Sunday = 7.
            4                 is_weekend          Calendar            Context   INTEGER                                                                    Indi

In [9]:
# =========================================================
# FEATURE DICTIONARY CONTENT VALIDATION
# =========================================================

required_metadata = [
    "feature_name",
    "category",
    "feature_role",
    "description",
    "calculation",
    "time_scope",
    "source_columns"
]


validation_rows = []

for column in required_metadata:

    missing_count = (
        feature_dictionary[column]
        .isna()
        .sum()
    )

    blank_count = (
        feature_dictionary[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    validation_rows.append({
        "metadata_field": column,
        "missing_values": int(missing_count),
        "blank_values": int(blank_count),
        "total_invalid": int(
            missing_count + blank_count
        )
    })


validation_df = pd.DataFrame(
    validation_rows
)


print("=" * 100)
print("FEATURE DICTIONARY CONTENT VALIDATION")
print("=" * 100)

print(
    validation_df.to_string(
        index=False
    )
)


assert (
    validation_df[
        "total_invalid"
    ] == 0
).all()

assert len(
    feature_dictionary
) == 85


print(
    "\n✅ SUCCESS — Every feature has complete semantic documentation."
)

FEATURE DICTIONARY CONTENT VALIDATION
metadata_field  missing_values  blank_values  total_invalid
  feature_name               0             0              0
      category               0             0              0
  feature_role               0             0              0
   description               0             0              0
   calculation               0             0              0
    time_scope               0             0              0
source_columns               0             0              0

✅ SUCCESS — Every feature has complete semantic documentation.


In [11]:
# =========================================================
# FEATURE DICTIONARY QUALITY CHECKS
# =========================================================

duplicate_features = (
    feature_dictionary[
        "feature_name"
    ]
    .duplicated()
    .sum()
)


missing_data_types = (
    feature_dictionary[
        "data_type"
    ]
    .isna()
    .sum()
)


missing_categories = (
    feature_dictionary[
        "category"
    ]
    .isna()
    .sum()
)


short_descriptions = (
    feature_dictionary[
        "description"
    ]
    .fillna("")
    .str.len()
    .lt(15)
    .sum()
)


expected_order = list(
    range(
        1,
        86
    )
)

actual_order = (
    feature_dictionary[
        "column_order"
    ]
    .tolist()
)


print("=" * 100)
print("FEATURE DICTIONARY QUALITY CHECKS")
print("=" * 100)

print(
    "Duplicate features  :",
    duplicate_features
)

print(
    "Missing data types  :",
    missing_data_types
)

print(
    "Missing categories  :",
    missing_categories
)

print(
    "Short descriptions  :",
    short_descriptions
)

print(
    "Column order valid  :",
    actual_order == expected_order
)


assert duplicate_features == 0
assert missing_data_types == 0
assert missing_categories == 0
assert short_descriptions == 0
assert actual_order == expected_order


print(
    "\n✅ SUCCESS — Feature Dictionary quality checks passed."
)

FEATURE DICTIONARY QUALITY CHECKS
Duplicate features  : 0
Missing data types  : 0
Missing categories  : 0
Short descriptions  : 0
Column order valid  : True

✅ SUCCESS — Feature Dictionary quality checks passed.


In [13]:
# =========================================================
# EXPORT FEATURE DICTIONARY
# =========================================================

feature_dictionary.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


assert OUTPUT_FILE.exists()


print("=" * 100)
print("FEATURE DICTIONARY EXPORT")
print("=" * 100)

print("Saved:")
print(OUTPUT_FILE)

print(
    "\nRows:",
    len(feature_dictionary)
)

print(
    "Unique features:",
    feature_dictionary[
        "feature_name"
    ].nunique()
)

print(
    "Columns in dictionary:",
    len(feature_dictionary.columns)
)


assert len(feature_dictionary) == 85

assert (
    feature_dictionary[
        "feature_name"
    ].nunique()
    == 85
)


print(
    "\n✅ SUCCESS — feature_dictionary_r62.csv exported."
)

FEATURE DICTIONARY EXPORT
Saved:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\results\feature_dictionary_r62.csv

Rows: 85
Unique features: 85
Columns in dictionary: 12

✅ SUCCESS — feature_dictionary_r62.csv exported.


In [15]:
# =========================================================
# FEATURE CATEGORY SUMMARY
# =========================================================

category_summary = (
    feature_dictionary
    .groupby(
        [
            "category",
            "feature_role"
        ]
    )
    .size()
    .reset_index(
        name="feature_count"
    )
    .sort_values(
        [
            "category",
            "feature_role"
        ]
    )
)


print("=" * 100)
print("FEATURE DICTIONARY CATEGORY SUMMARY")
print("=" * 100)

print(
    category_summary.to_string(
        index=False
    )
)

print(
    "\nTotal documented columns:",
    category_summary[
        "feature_count"
    ].sum()
)


assert (
    category_summary[
        "feature_count"
    ].sum()
    == 85
)


print(
    "\n✅ SUCCESS — Feature Dictionary category summary validated."
)

FEATURE DICTIONARY CATEGORY SUMMARY
         category       feature_role  feature_count
         Baseline      User Baseline             23
Baseline-Relative  Anomaly Indicator              4
Baseline-Relative Relative Magnitude              4
         Behavior      Daily Derived              3
         Calendar            Context              2
         Calendar                Key              1
    Cross-Channel      Daily Derived              1
    Cross-Channel   Rolling Behavior              1
            Email     Observed Daily             11
             HTTP     Observed Daily              6
       Identifier                Key              1
            Logon     Observed Daily              4
         Temporal   Rolling Behavior              9
         Temporal     Rolling Volume              6
         USB/File     Observed Daily              9

Total documented columns: 85

✅ SUCCESS — Feature Dictionary category summary validated.


In [17]:
# =========================================================
# FINAL FEATURE DICTIONARY PREVIEW
# =========================================================

preview_columns = [
    "column_order",
    "feature_name",
    "category",
    "feature_role",
    "data_type",
    "description",
    "calculation",
    "time_scope",
    "source_columns",
    "notes"
]


print("=" * 140)
print("FEATURE DICTIONARY PREVIEW")
print("=" * 140)

print(
    feature_dictionary[
        preview_columns
    ]
    .head(15)
    .to_string(index=False)
)


print("\n" + "=" * 140)

print(
    "Total features documented:",
    len(feature_dictionary)
)

print(
    "Output file:",
    OUTPUT_FILE
)

print("=" * 140)

print(
    "\n✅ FEATURE DICTIONARY R6.2 — COMPLETE"
)

FEATURE DICTIONARY PREVIEW
 column_order               feature_name   category   feature_role data_type                                                                           description                             calculation time_scope         source_columns                                                                             notes
            1                       user Identifier            Key   VARCHAR                                                        Unique identifier of the user.               Original user identifier. User level                   user                                                                                  
            2                        day   Calendar            Key      DATE                                   Calendar day used to aggregate the user's activity.                 CAST(timestamp AS DATE)      Daily       date / timestamp                                                                                  
            3   